In [2]:
import pandas as pd

# Load datasets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Check datasets
print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())
display(test.head())

print("Fraud rate:", train["is_fraud"].mean())

Train shape: (20000, 11)
Test shape: (10000, 10)


,claim_id,claim_amount,driver_age,previous_claim_count,vehicle_value,damage_severity,incident_type,repair_shop_id,phone_hash,address_id,is_fraud
0,C000001,254.29,42.0,2,11900.0,low,collision,Sb7a282e399,Pfd09f60f9e,A48656b5517,0
1,C000002,592.05,29.0,0,14600.0,low,collision,S9399315447,P8a7ff077bb,A53e94c25e5,0
2,C000007,980.14,58.0,0,17550.0,low,glass,S1bb582e486,P02246ac9cc,A84ed465d68,0
3,C000008,729.04,44.0,0,18550.0,low,collision,S7e56cdce9e,P8e162b8d1a,A46a2d8b11d,0
4,C000009,16145.05,34.0,0,29300.0,NaN,theft,S1bb582e486,Pd299199a4f,Ac638621b9b,0


,claim_id,claim_amount,driver_age,previous_claim_count,vehicle_value,damage_severity,incident_type,repair_shop_id,phone_hash,address_id
0,C000003,757.53,29.0,0,19700.0,medium,glass,S9dd55af075,Pf78d7edda7,Ae9e2783b3d
1,C000004,903.64,47.0,0,16450.0,low,collision,S71285760c2,P76b9d13e6a,A2dd867ebb2
2,C000005,20604.38,59.0,0,24750.0,high,theft,S1bb582e486,P7d5a46ed58,A659da9fba9
3,C000006,710.33,30.0,0,19650.0,low,collision,S1bb582e486,P41a8d51c6a,Ad4d7ace23a
4,C000010,6674.53,37.0,0,20600.0,high,collision,S19815489c9,P430e77acfb,Aafa83427b8


Fraud rate: 0.1717


In [3]:
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Separate features and target
X = train.drop(columns=["is_fraud", "claim_id"])
y = train["is_fraud"]

# Identify categorical columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# Handle missing categorical values
X = X.copy()
for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)

# Split training and validation data
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train CatBoost
model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=100,
    random_seed=42,
    cat_features=cat_cols
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)

# Predict fraud probabilities
val_probs = model.predict_proba(X_val)[:, 1]

print("Validation ROC-AUC:", roc_auc_score(y_val, val_probs))

0:	test: 0.6008651	best: 0.6008651 (0)	total: 73.2ms	remaining: 36.5s
100:	test: 0.8109103	best: 0.8109103 (100)	total: 1.09s	remaining: 4.31s
200:	test: 0.8144973	best: 0.8144973 (200)	total: 2.35s	remaining: 3.49s
300:	test: 0.8148461	best: 0.8148461 (300)	total: 4.2s	remaining: 2.78s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8149027847
bestIteration = 302

Shrink model to first 303 iterations.
Validation ROC-AUC: 0.8149027847160255


In [4]:
from sklearn.metrics import accuracy_score, classification_report

val_preds = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

Accuracy: 0.85775
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      3313
           1       0.69      0.31      0.43       687

    accuracy                           0.86      4000
   macro avg       0.78      0.64      0.67      4000
weighted avg       0.84      0.86      0.83      4000



In [5]:
import numpy as np

# Prepare test features
X_test = test.drop(columns=["claim_id"]).copy()

for col in cat_cols:
    X_test[col] = X_test[col].fillna("Missing").astype(str)

# Predict fraud probabilities
test_probs = model.predict_proba(X_test)[:, 1]

# Calculate expected financial benefit
expected_savings = test_probs * test["claim_amount"] - 2500

# Select up to 2000 claims with highest positive expected savings
investigate = np.zeros(len(test), dtype=int)

top_indices = np.argsort(-expected_savings.to_numpy())[:2000]

top_indices = [
    i for i in top_indices
    if expected_savings.iloc[i] > 0
]

investigate[top_indices] = 1

# Create submission
submission = pd.DataFrame({
    "claim_id": test["claim_id"],
    "investigate": investigate
})

submission.to_csv("submission.csv", index=False)

print("Claims investigated:", submission["investigate"].sum())
print("Submission shape:", submission.shape)

display(submission.head())

Claims investigated: 1145
Submission shape: (10000, 2)


,claim_id,investigate
0,C000003,0
1,C000004,0
2,C000005,0
3,C000006,0
4,C000010,0


In [6]:
import numpy as np
import pandas as pd

# Get claim amounts and actual fraud labels
val_amounts = train.loc[X_val.index, "claim_amount"].to_numpy()
val_fraud = y_val.to_numpy()

# Calculate expected savings
expected_savings = val_probs * val_amounts - 2500

# Validation set contains 4000 claims
# Scale the 2000/10000 investigation limit to 800/4000
budget = int(len(X_val) * 0.2)

# Select claims with the highest positive expected savings
investigate = np.zeros(len(X_val), dtype=int)

top_indices = np.argsort(-expected_savings)[:budget]

top_indices = [
    i for i in top_indices
    if expected_savings[i] > 0
]

investigate[top_indices] = 1

# Calculate actual competition score
score = np.sum(
    investigate * (val_fraud * val_amounts - 2500)
)

# Calculate additional metrics
n_investigated = investigate.sum()
frauds_detected = np.sum(investigate * val_fraud)
total_fraud = val_fraud.sum()

print(f"Validation financial score: ${score:,.2f}")
print(f"Claims investigated: {n_investigated}")
print(f"Fraudulent claims detected: {frauds_detected}")
print(f"Total fraudulent claims: {total_fraud}")
print(f"Fraud recall: {frauds_detected / total_fraud:.2%}")

Validation financial score: $2,387,600.40
Claims investigated: 476
Fraudulent claims detected: 195
Total fraudulent claims: 687
Fraud recall: 28.38%


In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np

# Prepare features
X = train.drop(columns=["is_fraud", "claim_id"]).copy()
y = train["is_fraud"]

cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)

# 5-fold cross-validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_probs = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    model_cv = CatBoostClassifier(
        iterations=303,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        verbose=False,
        random_seed=42,
        cat_features=cat_cols
    )

    model_cv.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    oof_probs[val_idx] = model_cv.predict_proba(
        X.iloc[val_idx]
    )[:, 1]

    print(f"Fold {fold} completed")

# Calculate expected savings
amounts = train["claim_amount"].to_numpy()
fraud = y.to_numpy()

expected_savings = oof_probs * amounts - 2500

# Scale investigation budget to 20,000 claims
budget = 4000

investigate = np.zeros(len(train), dtype=int)

top_indices = np.argsort(-expected_savings)[:budget]
top_indices = top_indices[expected_savings[top_indices] > 0]

investigate[top_indices] = 1

# Calculate S score
S = np.sum(
    investigate * (fraud * amounts - 2500)
)

print("\n===== CROSS-VALIDATION RESULTS =====")
print(f"ROC-AUC: {roc_auc_score(y, oof_probs):.4f}")
print(f"Claims investigated: {investigate.sum()}")
print(f"Frauds detected: {np.sum(investigate * fraud)}")
print(f"S Score: ${S:,.2f}")
print(f"Estimated S for 10,000 claims: ${S / 2:,.2f}")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed

===== CROSS-VALIDATION RESULTS =====
ROC-AUC: 0.8036
Claims investigated: 2406
Frauds detected: 982
S Score: $11,944,392.08
Estimated S for 10,000 claims: $5,972,196.04


In [8]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

,Feature,Importance
7,phone_hash,31.321669
2,previous_claim_count,14.143908
6,repair_shop_id,13.423298
8,address_id,11.451084
0,claim_amount,9.412049
1,driver_age,8.043310
5,incident_type,5.532323
3,vehicle_value,3.861927
4,damage_severity,2.810432


In [9]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. Prepare data
# ==========================================

X = train.drop(columns=["is_fraud", "claim_id"]).copy()
y = train["is_fraud"]

cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)


# ==========================================
# 2. Feature engineering
# ==========================================

def add_features(X_train_fold, X_val_fold):

    X_train_fold = X_train_fold.copy()
    X_val_fold = X_val_fold.copy()

    # Feature 1: Claim-to-vehicle value ratio
    for df in [X_train_fold, X_val_fold]:
        df["claim_vehicle_ratio"] = (
            df["claim_amount"] /
            df["vehicle_value"].replace(0, np.nan)
        )

    # Features 2-4: Identifier frequencies
    for col in ["phone_hash", "repair_shop_id", "address_id"]:

        # Learn frequencies from training fold only
        counts = X_train_fold[col].value_counts()

        X_train_fold[col + "_frequency"] = (
            X_train_fold[col].map(counts).fillna(0)
        )

        X_val_fold[col + "_frequency"] = (
            X_val_fold[col].map(counts).fillna(0)
        )

    return X_train_fold, X_val_fold


# ==========================================
# 3. 5-fold cross-validation
# ==========================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_probs_fe = np.zeros(len(train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    X_tr = X.iloc[train_idx]
    X_va = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]

    # Add engineered features
    X_tr, X_va = add_features(X_tr, X_va)

    # Train CatBoost
    model_fe = CatBoostClassifier(
        iterations=303,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        verbose=False,
        random_seed=42,
        cat_features=cat_cols
    )

    model_fe.fit(X_tr, y_tr)

    # Predict validation probabilities
    oof_probs_fe[val_idx] = model_fe.predict_proba(
        X_va
    )[:, 1]

    print(f"Fold {fold} completed")


# ==========================================
# 4. Calculate competition S score
# ==========================================

amounts = train["claim_amount"].to_numpy()
fraud = y.to_numpy()

# Expected savings
expected_savings = oof_probs_fe * amounts - 2500

# Maximum 20% of claims investigated
budget = 4000

investigate = np.zeros(len(train), dtype=int)

top_indices = np.argsort(-expected_savings)[:budget]

# Only investigate positive expected savings
top_indices = top_indices[
    expected_savings[top_indices] > 0
]

investigate[top_indices] = 1

# Calculate actual financial score
S = np.sum(
    investigate * (fraud * amounts - 2500)
)

# ==========================================
# 5. Display results
# ==========================================

print("\n===== FEATURE ENGINEERING RESULTS =====")

print(f"ROC-AUC: {roc_auc_score(y, oof_probs_fe):.4f}")
print(f"Claims investigated: {investigate.sum()}")
print(f"Frauds detected: {np.sum(investigate * fraud)}")
print(f"S Score: ${S:,.2f}")
print(f"Estimated S for 10,000 claims: ${S / 2:,.2f}")

Fold 1 completed
Fold 2 completed
Fold 3 completed
Fold 4 completed
Fold 5 completed

===== FEATURE ENGINEERING RESULTS =====
ROC-AUC: 0.8117
Claims investigated: 2201
Frauds detected: 968
S Score: $12,190,989.48
Estimated S for 10,000 claims: $6,095,494.74


In [10]:
# ==========================================
# 1. Prepare full training and test data
# ==========================================

X_full = train.drop(columns=["is_fraud", "claim_id"]).copy()
X_test = test.drop(columns=["claim_id"]).copy()

y_full = train["is_fraud"]

# Handle categorical missing values
for col in cat_cols:
    X_full[col] = X_full[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

# ==========================================
# 2. Apply feature engineering
# ==========================================

X_full, X_test = add_features(X_full, X_test)

# ==========================================
# 3. Train final CatBoost model
# ==========================================

final_model = CatBoostClassifier(
    iterations=303,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    verbose=100,
    random_seed=42,
    cat_features=cat_cols
)

final_model.fit(X_full, y_full)

# ==========================================
# 4. Predict test fraud probabilities
# ==========================================

test_probs = final_model.predict_proba(X_test)[:, 1]

# ==========================================
# 5. Calculate expected financial savings
# ==========================================

expected_savings = (
    test_probs * test["claim_amount"].to_numpy() - 2500
)

# Select at most 2000 claims
budget = 2000

investigate = np.zeros(len(test), dtype=int)

top_indices = np.argsort(-expected_savings)[:budget]

top_indices = top_indices[
    expected_savings[top_indices] > 0
]

investigate[top_indices] = 1

# ==========================================
# 6. Generate submission
# ==========================================

submission = pd.DataFrame({
    "claim_id": test["claim_id"],
    "investigate": investigate
})

submission.to_csv("submission.csv", index=False)

print("\n===== FINAL SUBMISSION =====")
print("Submission shape:", submission.shape)
print("Claims investigated:", submission["investigate"].sum())
print("Unique claim IDs:", submission["claim_id"].nunique())

display(submission.head())

0:	learn: 0.6578068	total: 29.5ms	remaining: 8.9s
100:	learn: 0.3448270	total: 2.65s	remaining: 5.3s
200:	learn: 0.3320455	total: 4.53s	remaining: 2.3s
300:	learn: 0.3221649	total: 6.32s	remaining: 42ms
302:	learn: 0.3220314	total: 6.35s	remaining: 0us

===== FINAL SUBMISSION =====
Submission shape: (10000, 2)
Claims investigated: 998
Unique claim IDs: 10000


,claim_id,investigate
0,C000003,0
1,C000004,0
2,C000005,0
3,C000006,0
4,C000010,0
